<a href="https://colab.research.google.com/github/nicobargioni/issd_ph_2026/blob/main/tp1-pdf/TP1_procesamiento_PDF_Bargioni.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!-- 🧠 Prompt: "armá la portada del notebook con título, alumno, fuente elegida, objetivo y el diagrama del pipeline" -->

# TP1 — Procesamiento de archivos PDF
### Procesamiento del Habla (PH) · ISSD · 4° IAR

**Alumno:** Nicolás Bargioni

**Fuente elegida:** *Panorama económico de América Latina y el Caribe — Revisitando la política industrial* (Oficina del Economista Jefe para América Latina y el Caribe, **Banco Mundial**, abril 2026).
URL: https://openknowledge.worldbank.org/server/api/core/bitstreams/a1baf8eb-b679-43f6-b49b-a97d29838bd8/content

> Documento en **español**, con metadatos, **tablas estadísticas** (proyecciones de PIB por país) y abundante **texto narrativo** — ideal para recorrer todo el pipeline de adquisición y limpieza visto en clase.

---
## Objetivo
Construir un flujo en Python que transforme un PDF en datos estructurados y listos para análisis, aplicando la **etapa 1 del pipeline de PH** (texto crudo → limpieza), con las herramientas vistas en clase:
`requests` · `PyMuPDF (fitz)` · `pdfplumber` · `NLTK` · `spaCy` · `pandas` · `matplotlib`.

```
PDF (raw data) → metadatos → texto crudo → tabla (DataFrame) → tokenización → stopwords → frecuencias
```

<!-- 🧠 Prompt: "escribí una intro corta para la sección de instalación de dependencias" -->

## 0. Preparación del entorno

Instalamos e importamos las librerías. En Colab algunas ya vienen; instalamos las que faltan.

In [ ]:
# 🧠 Prompt: "instalá las dependencias necesarias en Colab (requests, pymupdf, pdfplumber, nltk, spacy)"
# Instalación (Colab). Si ya están instaladas, pip no hace nada.
!pip -q install pymupdf pdfplumber nltk spacy
!python -m spacy download es_core_news_sm -q   # modelo español de spaCy (visto en clase)


In [ ]:
# 🧠 Prompt: "importá las librerías y descargá los recursos de NLTK y el modelo de spaCy"
import requests                     # descarga de archivos (clase 1: scraping/adquisición)
import fitz                          # PyMuPDF: metadatos + texto crudo
import pdfplumber                    # extracción de tablas
import pandas as pd                  # DataFrame
import matplotlib.pyplot as plt      # visualización
import re, string
from collections import Counter      # conteo de frecuencias

import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer

# Recursos de NLTK (tokenizador y stopwords). 'punkt_tab' es el nuevo punkt.
for rec in ["punkt", "punkt_tab", "stopwords"]:
    try:
        nltk.download(rec, quiet=True)
    except Exception as e:
        print("aviso descargando", rec, e)

import spacy
nlp = spacy.load("es_core_news_sm")
print("Entorno listo ✔")


<!-- 🧠 Prompt: "explicá en markdown el paso de adquisición y por qué se usa un User-Agent para evitar el 403" -->

## 1. Adquisición y metadatos

### 1.1 Descarga automatizada
Usamos `requests` con un **User-Agent de navegador**. En clase vimos que muchos servidores responden **403** a un script "pelado": simular Chrome con headers evita el bloqueo.

In [ ]:
# 🧠 Prompt: "descargá el archivo/página con requests usando headers de navegador y mostrá el status code"
URL = "https://openknowledge.worldbank.org/server/api/core/bitstreams/a1baf8eb-b679-43f6-b49b-a97d29838bd8/content"
ARCHIVO = "panorama_lac_bm.pdf"

headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                         "AppleWebKit/537.36 (KHTML, like Gecko) "
                         "Chrome/124.0 Safari/537.36"}

resp = requests.get(URL, headers=headers, timeout=90)
print("Status code:", resp.status_code)          # 200 = OK
print("Content-Type:", resp.headers.get("Content-Type"))
print("Tamaño:", round(len(resp.content)/1_000_000, 2), "MB")

with open(ARCHIVO, "wb") as f:
    f.write(resp.content)
print("Guardado como", ARCHIVO)


<!-- 🧠 Prompt: "redactá la explicación de cómo inspeccionar los metadatos del PDF y qué mostrar" -->

### 1.2 Inspección de metadatos
Abrimos el PDF con **PyMuPDF (`fitz`)** y mostramos los metadatos pedidos: **título, autor, número de páginas y software de creación**.

In [ ]:
# 🧠 Prompt: "abrí el PDF con PyMuPDF y mostrá título, autor, páginas y software de creación"
doc = fitz.open(ARCHIVO)
meta = doc.metadata

print("Título   :", repr(meta.get("title")))
print("Autor    :", repr(meta.get("author")))
print("N° páginas:", doc.page_count)
print("Creador  :", meta.get("creator"))      # software de creación
print("Productor:", meta.get("producer"))
print("Creado   :", meta.get("creationDate"))


<!-- 🧠 Prompt: "redactá la explicación de cómo inspeccionar los metadatos del PDF y qué mostrar" -->

**Análisis de los metadatos.**
El PDF **sí** declara el *software de creación* (`Adobe InDesign 21.3 (Macintosh)`) y el *productor*, pero **los campos `title` y `author` vienen vacíos**. Esto es muy común: los metadatos internos *no siempre están completos* y no hay que confiar ciegamente en ellos. Cuando el dato es importante, conviene **recuperarlo del propio contenido** (la portada). Lo hacemos extrayendo el texto de la primera página:

In [ ]:
# 🧠 Prompt: "extraé el texto de la portada para reconstruir el título efectivo"
portada = doc[0].get_text()
print("----- Texto de la portada (pág. 1) -----")
print(portada.strip()[:400])

# Reconstruimos un "título efectivo" desde la portada
titulo_efectivo = " ".join(portada.split("\n")[0:3]).strip()
print("\nTítulo efectivo (desde portada):", titulo_efectivo)


<!-- 🧠 Prompt: "explicá el análisis estructural del texto crudo y los caracteres separadores" -->

## 2. Análisis estructural del texto

### 2.1 Extracción cruda de las primeras 5 páginas
La función de PyMuPDF que devuelve el texto **lo más crudo posible** es `page.get_text()` (modo `"text"` por defecto): respeta los saltos de línea tal como los maquetó el PDF, sin reordenar.

In [ ]:
# 🧠 Prompt: "extraé el texto crudo de las primeras 5 páginas y mostralo con repr()"
paginas_crudas = []
for i in range(5):                       # primeras 5 páginas (índices 0–4)
    paginas_crudas.append(doc[i].get_text())

texto_5 = "\n".join(paginas_crudas)
print("Caracteres extraídos (5 págs):", len(texto_5))
print("\n----- Muestra CRUDA con repr() para ver los separadores reales -----\n")
print(repr(texto_5[:600]))      # repr() hace visibles \n, \t, etc.


<!-- 🧠 Prompt: "explicá el análisis estructural del texto crudo y los caracteres separadores" -->

### 2.2 Análisis de separadores
Observamos la salida cruda (con `repr()`, que muestra los caracteres invisibles) para describir cómo se separan párrafos, secciones y capítulos.

In [ ]:
# 🧠 Prompt: "contá los distintos caracteres separadores (\n, \t, \r, espacios) en el texto crudo"
conteo = {
    "\\n  (salto de línea)": texto_5.count("\n"),
    "\\n\\n (línea en blanco = párrafo)": texto_5.count("\n\n"),
    "\\t  (tabulación)": texto_5.count("\t"),
    "\\r  (retorno de carro)": texto_5.count("\r"),
    "  (doble espacio)": texto_5.count("  "),
    "\\x0c (form feed / salto de página)": texto_5.count("\x0c"),
}
for k, v in conteo.items():
    print(f"{k:42}: {v}")


<!-- 🧠 Prompt: "explicá el análisis estructural del texto crudo y los caracteres separadores" -->

**Respuesta — ¿qué caracteres funcionan como separadores en este PDF?**

- **`\n` (salto de línea):** es el separador dominante. PyMuPDF inserta un `\n` **al final de cada línea visual** del PDF (no por oración). Por eso una misma frase aparece partida en varias líneas: el salto refleja el **renglón de maquetación**, no la gramática.
- **`\n\n` (doble salto):** aparece entre **bloques/párrafos** y entre secciones — es la pista más útil para separar párrafos.
- **`\t` y `\r`:** prácticamente **no se usan** en este archivo (conteo ≈ 0). No son separadores relevantes acá.
- **Secuencias de espacios (`  `):** aparecen por la **maquetación a columnas/justificado**; son ruido a normalizar.
- **Capítulos/secciones:** no hay un carácter especial que los delimite; se reconocen por **títulos cortos en línea propia** seguidos de `\n` (ej. *"Perspectivas de crecimiento para la región"*).

**Conclusión:** el texto crudo **no respeta la unidad "oración/párrafo"**, sino la **línea visual**. Para limpiar habría que **re-unir líneas** (reemplazar `\n` simples por espacio y conservar `\n\n` como separador de párrafo) — un caso típico de trabajo de *data janitor*.

<!-- 🧠 Prompt: "describí cómo extraer una tabla del PDF con pdfplumber y pasarla a DataFrame" -->

## 3. Extracción de datos tabulares

Buscamos una página con una tabla de datos. La pág. **20** (índice 19) contiene *"Tasas de crecimiento del PIB real"* por país y año — una tabla limpia y bien estructurada. Usamos **`pdfplumber`** (sugerido en la consigna).

In [ ]:
# 🧠 Prompt: "extraé la tabla de la página elegida con pdfplumber"
PAGINA_TABLA = 19  # índice 0-based -> página 20 del documento

with pdfplumber.open(ARCHIVO) as pdf:
    page = pdf.pages[PAGINA_TABLA]
    tablas = page.extract_tables()

print("Tablas detectadas en la página:", len(tablas))
tabla = tablas[0]
print("Dimensiones:", len(tabla), "filas x", len(tabla[0]), "columnas")
for fila in tabla[:6]:
    print(fila)


<!-- 🧠 Prompt: "detallá la limpieza de la tabla: encabezados, coma decimal a float y filas vacías" -->

### 3.1 A DataFrame de Pandas + limpieza básica
- La **primera fila** son los encabezados (años); la usamos como `columns`.
- La primera columna es el país → la nombramos `"País"`.
- Eliminamos **filas vacías**.
- Los números vienen con **coma decimal** (formato español) → los pasamos a `float` con punto.

In [ ]:
# 🧠 Prompt: "convertí la tabla a DataFrame y limpiala (encabezados, coma decimal, filas vacías)"
# 1) separar encabezado y cuerpo
encabezado = tabla[0]
cuerpo = tabla[1:]

df = pd.DataFrame(cuerpo, columns=encabezado)

# 2) nombrar la primera columna (venía como '')
df = df.rename(columns={df.columns[0]: "País"})

# 3) eliminar filas totalmente vacías o sin país
df["País"] = df["País"].astype(str).str.strip()
df = df[df["País"].ne("") & df["País"].ne("None")].reset_index(drop=True)

# 4) convertir columnas de años a número (coma decimal -> punto)
cols_anios = [c for c in df.columns if c != "País"]
for c in cols_anios:
    df[c] = (df[c].astype(str)
                  .str.replace(",", ".", regex=False)
                  .str.replace("−", "-", regex=False)   # signo menos unicode
                  .replace({"": None, "None": None}))
    df[c] = pd.to_numeric(df[c], errors="coerce")

print("Shape final:", df.shape)
df.head(10)


In [ ]:
# 🧠 Prompt: "filtrá y mostrá la fila de Argentina como verificación"
# Pequeña verificación: la fila de Argentina
df[df["País"].str.contains("Argentina", case=False, na=False)]


<!-- 🧠 Prompt: "describí cómo extraer una tabla del PDF con pdfplumber y pasarla a DataFrame" -->

**Comentario.** Quedó una tabla limpia: filas = países, columnas = años (`2022`–`2027p`, donde `e`=estimado y `p`=proyectado), valores numéricos de crecimiento del PIB real. Ya es un `DataFrame` analizable (se podría graficar la serie de Argentina, ordenar por crecimiento, etc.).

<!-- 🧠 Prompt: "introducí el análisis de frecuencias y la limpieza con NLTK" -->

## 4. Análisis de frecuencias (NLP básico)

Elegimos una página con **texto narrativo** (pág. 13, índice 12) y aplicamos la limpieza de la **etapa 1 del pipeline**: tokenización → minúsculas → quitar stopwords y puntuación → frecuencias.

In [ ]:
# 🧠 Prompt: "obtené el texto crudo de toda la página con get_text() y mostrá una muestra"
PAGINA_TEXTO = 12  # índice 0-based -> página 13
texto = doc[PAGINA_TEXTO].get_text()
print(texto[:500], "...")


<!-- 🧠 Prompt: "explicá la tokenización en español y por qué declarar el idioma" -->

### 4.1 Tokenización + minúsculas
Tokenizamos **declarando el idioma español** (en clase se remarcó que tokenizar en español ≠ en inglés).

In [ ]:
# 🧠 Prompt: "tokenizá el texto en minúsculas declarando idioma español"
texto_min = texto.lower()
tokens = word_tokenize(texto_min, language="spanish")
print("Total de tokens (crudos):", len(tokens))
print("Muestra:", tokens[:20])


<!-- 🧠 Prompt: "explicá cómo se quitan stopwords y puntuación antes de contar frecuencias" -->

### 4.2 Limpieza: stopwords + puntuación
Quitamos las **stopwords del español** (NLTK) y la puntuación. También descartamos tokens que sean sólo números o símbolos, que no aportan al análisis léxico.

In [ ]:
# 🧠 Prompt: "filtrá stopwords del español, puntuación, números y tokens cortos"
stop_es = set(stopwords.words("spanish"))
puntuacion = set(string.punctuation) | {"…", "”", "“", "«", "»", "—", "–", "‘", "’", "º", "%"}

def es_palabra_util(tok):
    if tok in stop_es:           return False   # stopword
    if tok in puntuacion:        return False   # signo suelto
    if len(tok) <= 2:            return False   # tokens muy cortos (de, el, ya cubiertos; resto ruido)
    if re.fullmatch(r"[\d.,;:%/()\-]+", tok):  return False  # números/símbolos
    return True

tokens_limpios = [t for t in tokens if es_palabra_util(t)]
print("Tokens tras limpieza:", len(tokens_limpios))
print("Muestra:", tokens_limpios[:25])


<!-- 🧠 Prompt: "pedí un gráfico de barras con las 15 palabras más frecuentes" -->

### 4.3 Visualización: 15 palabras más frecuentes

In [ ]:
# 🧠 Prompt: "calculá las 15 palabras más frecuentes con Counter"
frec = Counter(tokens_limpios)
top15 = frec.most_common(15)
print(top15)

palabras = [p for p, _ in top15]
valores  = [c for _, c in top15]

plt.figure(figsize=(10, 5))
plt.barh(palabras[::-1], valores[::-1])          # horizontal, de mayor a menor
plt.title("Top 15 palabras más frecuentes — Panorama económico LAC (pág. 13)")
plt.xlabel("Frecuencia")
plt.tight_layout()
plt.show()


<!-- 🧠 Prompt: "explicá cómo se quitan stopwords y puntuación antes de contar frecuencias" -->

### 4.4 ¿Aparecen tokens que también considerarías *stopwords*? — análisis

**Sí.** Aunque ya filtramos las stopwords genéricas del español, en el ranking quedan palabras que **en este dominio** funcionan como stopwords (alta frecuencia, bajo poder discriminante). Mirando el top-15 de esta página:

- **Stopwords de dominio:** *"crecimiento"* (y en general *"condiciones", "precios", "economías"*) es tan transversal a TODO un informe económico que **no distingue un párrafo de otro**. Conviene sumarlas a una **lista de stopwords personalizada del dominio** (igual que en clase: en un discurso político *"gobierno"* puede ser stopword de dominio).
- **Conectores/poco informativos que NLTK NO trae:** aparecen tokens como *"mientras", "medida", "medio"* — son de bajo contenido pero **no están en la lista de stopwords del español de NLTK**, así que pasan el filtro. Esto demuestra que **la lista por defecto no es exhaustiva** y hay que ampliarla a mano.
- **Matiz de clase (auxiliares):** formas de *"ser"/"estar"* pueden colarse. spaCy trata `estar` como **auxiliar/stopword** en *"están creciendo"*, pero NO en *"estar en recesión"*: la decisión **depende del contexto**, no es automática.

**Justificación / conclusión.** La lista de stopwords **no es universal**: depende del **idioma** *y* del **dominio**, e incluso la lista estándar deja afuera conectores como *"mientras"*. Una segunda pasada con stopwords de dominio (`crecimiento`, `precios`, `condiciones`, `mientras`…) dejaría ver mejor los temas distintivos del texto (p. ej. *inflación*, *política monetaria*, *incertidumbre*).

<!-- 🧠 Prompt: "compará stemming (NLTK) vs lematización (spaCy) como recurso de clase" -->

### 4.5 (Extra) Stemming vs. Lematización — recurso de clase
Para reforzar lo visto en clase, comparamos cómo **NLTK SnowballStemmer** (raíz bruta) y **spaCy** (lema = forma de diccionario) tratan las palabras top.

In [ ]:
# 🧠 Prompt: "compará stemming de NLTK con la lematización de spaCy sobre las palabras top"
stemmer = SnowballStemmer("spanish")
muestra = palabras[:8]

doc_spacy = nlp(" ".join(muestra))
lemas = {t.text: t.lemma_ for t in doc_spacy}

print(f"{'palabra':15}{'stemming (NLTK)':22}{'lema (spaCy)':18}")
print("-"*55)
for w in muestra:
    print(f"{w:15}{stemmer.stem(w):22}{lemas.get(w, w):18}")


<!-- 🧠 Prompt: "compará stemming (NLTK) vs lematización (spaCy) como recurso de clase" -->

**Lectura.** El *stemming* recorta de forma agresiva (ej. *económico → económ*), útil para **reducir el vocabulario** rápido pero genera raíces no-palabra. La *lematización* devuelve formas reales (ej. *países → país*), más interpretable pero más costosa. Cuál usar depende del objetivo: vectorizar/clasificar (stemming suele alcanzar) vs. análisis interpretable (lematización).

<!-- 🧠 Prompt: "explicá en markdown el paso de adquisición y por qué se usa un User-Agent para evitar el 403" -->

## 5. Conclusiones

- **Adquisición:** `requests` + User-Agent descargó el PDF sin bloqueo (status 200). Los **metadatos** estaban incompletos (sin título/autor) → se recuperó el título desde la **portada**: no hay que confiar ciegamente en los metadatos.
- **Estructura:** el separador dominante es **`\n`**, que marca la **línea visual** y no la oración; los párrafos se intuyen por **`\n\n`**. `\t`/`\r` no se usan. Limpiar implica **re-unir líneas**.
- **Tablas:** `pdfplumber` extrajo limpia la tabla de **crecimiento del PIB** (países × años); se normalizó la **coma decimal** y se nombraron columnas → `DataFrame` listo.
- **Frecuencias:** tras quitar stopwords + puntuación, el top está dominado por **stopwords de dominio** (*crecimiento, región, países*) → confirma que la lista de stopwords **depende del idioma y del dominio**.
- Se recorrió así la **etapa 1 del pipeline de PH** (texto crudo → limpieza) con las herramientas de clase.

## 6. Referencias
- **Fuente PDF:** Banco Mundial (2026). *Panorama económico de América Latina y el Caribe — Revisitando la política industrial*. Oficina del Economista Jefe para ALC. https://openknowledge.worldbank.org/server/api/core/bitstreams/a1baf8eb-b679-43f6-b49b-a97d29838bd8/content
- **PyMuPDF (fitz):** https://pymupdf.readthedocs.io/
- **pdfplumber:** https://github.com/jsvine/pdfplumber
- **NLTK:** https://www.nltk.org/  · **spaCy (es_core_news_sm):** https://spacy.io/models/es
- **Material de clase PH (ISSD):** pipeline texto, NLTK/spaCy, tokenización en español, stemming vs lematización (clases 1–2).
- **Asistencia IA:** _[completar: pegar acá el enlace a la conversación con la IA utilizada]_